# Notebook 13 — Hybrid production architecture

This final AgentOps notebook combines the earlier lessons into a production-oriented architecture. Instead of building one giant autonomous agent, we start with a deterministic workflow that classifies the task, selects the least autonomous reliable path, applies policy checks, and pauses for human approval before consequential actions.

The core principle is simple but easy to forget: **agents are components inside a system, not the system itself**.

## Architecture

```mermaid
flowchart TD
    W["Deterministic workflow"] --> C["Classify task"]
    C --> L["Simple lookup"]
    C --> I["Investigation"]
    C --> H["High-risk case"]
    L --> D["Deterministic status/report workflow"]
    I --> A["Single bounded agent"]
    H --> T["Specialist agent team"]
    D --> P["Policy checks"]
    A --> P
    T --> P
    P --> R{"High-impact action?"}
    R -- "no" --> F["Final recommendation"]
    R -- "yes" --> U["Human approval"]
    U --> X["Approved action"]
```

This is the architecture ladder from the article turned into an executable router.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo / "labs") not in sys.path:
    sys.path.insert(0, str(repo / "labs"))

from agentops_lab.hybrid_production_architecture import run_examples

plans = run_examples()
[(plan.route, plan.architecture, plan.approval_required) for plan in plans]


## Inspect the routing decisions

The router does not ask the model what architecture is fashionable. It uses operational features: whether the path is known, how ambiguous the evidence is, how risky the action is, and how large the customer impact is.

In [ ]:
for plan in plans:
    print(f"{plan.route}: {plan.architecture}")
    print(f"  reason: {plan.reason}")
    print(f"  checks: {', '.join(plan.policy_checks)}")
    print(f"  approval required: {plan.approval_required}\n")


## Design checklist

A credible production agent system should be able to answer these questions before launch:

1. What task classes are handled deterministically?
2. Which tasks justify a bounded single agent?
3. Which tasks justify a specialist team, and what measured improvement beats the overhead?
4. Which tools are read-only, which are propose-only, and which require approval?
5. What budgets stop loops before cost, latency, or risk becomes unacceptable?
6. What traces and receipts prove what happened after the run?

If those answers are missing, the system is not production-ready yet.

## Reflection

1. Where would you place LangGraph in this architecture?
2. Where would OpenAI Agents SDK be enough?
3. Where would AutoGen or CrewAI make the team easier to reason about?
4. Which decisions must stay deterministic even when agents are involved?